# 라우팅 워크플로 — Routing Workflows

**Skilljar Lesson 04 대응**

이 노트북에서 다루는 내용:
1. 분류기(Classifier) 구현
2. 전문 핸들러(Specialized Handlers) 구현
3. 2단계 라우팅 파이프라인
4. 건축공학 적용: 문서 유형별 분기 처리

In [ ]:
# ── Setup ──────────────────────────────────────────────
import anthropic
import json
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"

## §1. 분류기 (Classifier)

입력을 먼저 카테고리로 분류합니다. 분류는 비교적 단순한 작업이므로 저비용 모델을 사용합니다.

In [ ]:
def classify_input(user_message):
    """입력을 카테고리로 분류한다."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=50,
        system=(
            "Classify the following customer message into exactly one "
            "category: 'refund', 'technical', 'complaint', or 'general'. "
            "Respond with only the category name, nothing else."
        ),
        messages=[{"role": "user", "content": user_message}]
    )
    return response.content[0].text.strip().lower()

# 테스트
test_messages = [
    "I want a refund for my order #12345",
    "The app keeps crashing when I open it",
    "Your service is terrible and I'm very upset",
    "What are your business hours?"
]

for msg in test_messages:
    category = classify_input(msg)
    print(f"  [{category:>10}] {msg}")

## §2. 전문 핸들러 (Specialized Handlers)

In [ ]:
HANDLERS = {
    "refund": {
        "system": "You are a refund specialist. Help the customer with "
                  "their refund request. Be empathetic and follow the "
                  "30-day refund policy. Ask for order number if not provided. "
                  "Respond in Korean."
    },
    "technical": {
        "system": "You are a technical support engineer. Help diagnose "
                  "and resolve technical issues. Ask clarifying questions "
                  "and provide step-by-step solutions. Respond in Korean."
    },
    "complaint": {
        "system": "You are a customer relations specialist. Handle "
                  "complaints with empathy. Acknowledge the issue, "
                  "apologize, and offer concrete resolution steps. "
                  "Respond in Korean."
    },
    "general": {
        "system": "You are a helpful customer service agent. Answer "
                  "general questions clearly and concisely. Respond in Korean."
    }
}

print(f"핸들러 {len(HANDLERS)}개 정의 완료: {list(HANDLERS.keys())}")

## §3. 라우팅 파이프라인

In [ ]:
def route_and_handle(user_message):
    """입력을 분류하고 전문 핸들러로 라우팅한다."""
    # Step 1: 분류
    category = classify_input(user_message)
    print(f"  🏷️ 분류 결과: {category}")
    
    # Step 2: 핸들러 선택 및 처리
    handler = HANDLERS.get(category, HANDLERS["general"])
    
    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        system=handler["system"],
        messages=[{"role": "user", "content": user_message}]
    )
    
    return {
        "category": category,
        "response": response.content[0].text
    }

# 테스트
result = route_and_handle(
    "I bought a product 2 weeks ago and it's defective. I want my money back."
)
print(f"\n💬 응답:\n{result['response']}")

## §4. 건축공학 적용: 문서 유형별 라우팅

건설 문서를 유형(시방서/계약서/보고서)에 따라 다른 전문 분석을 수행합니다.

In [ ]:
# 건축공학 문서 라우팅
DOC_HANDLERS = {
    "specification": {
        "system": "You are a construction specification analyst. "
                  "Extract key material requirements, performance criteria, "
                  "and compliance standards. Respond in Korean."
    },
    "contract": {
        "system": "You are a construction contract analyst. "
                  "Identify key clauses, obligations, penalties, "
                  "and risk allocation. Respond in Korean."
    },
    "report": {
        "system": "You are a structural engineering report analyst. "
                  "Summarize findings, identify critical issues, "
                  "and extract recommendations. Respond in Korean."
    }
}

def classify_document(doc_text):
    """건설 문서를 유형별로 분류한다."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=50,
        system=(
            "Classify this construction document into exactly one category: "
            "'specification', 'contract', or 'report'. "
            "Respond with only the category name."
        ),
        messages=[{"role": "user", "content": doc_text[:500]}]
    )
    return response.content[0].text.strip().lower()

def route_document(doc_text):
    """건설 문서를 분류하고 전문 분석기로 라우팅한다."""
    doc_type = classify_document(doc_text)
    print(f"  📄 문서 유형: {doc_type}")
    
    handler = DOC_HANDLERS.get(doc_type, DOC_HANDLERS["report"])
    
    response = client.messages.create(
        model=MODEL,
        max_tokens=2048,
        system=handler["system"],
        messages=[{"role": "user", "content": f"Analyze this document:\n\n{doc_text}"}]
    )
    return {"type": doc_type, "analysis": response.content[0].text}

# 테스트: 시방서 예시
spec_doc = """
Section 03 30 00 - Cast-in-Place Concrete

1. General Requirements
   1.1 Concrete strength: f'c = 35 MPa at 28 days
   1.2 Slump: 150 ± 25 mm
   1.3 Air content: 4.5% ± 1.5%
   1.4 W/C ratio: Maximum 0.45

2. Reinforcement
   2.1 Grade: SD400 (fy = 400 MPa)
   2.2 Cover: 40mm (interior), 50mm (exterior)
"""

result = route_document(spec_doc)
print(f"\n📊 분석 결과:\n{result['analysis']}")

## §5. 핵심 정리

- **라우팅 = 분류 + 전문 처리**: 2단계 파이프라인
- **비용 효율**: 분류에 저비용 모델, 처리에 고성능 모델 사용 가능
- **확장성**: 새 카테고리 추가 시 핸들러만 추가
- **다음 노트북**: `S8_05_agent_tools.ipynb`에서 에이전트를 구현한다